# WLASL landmark accuracy chase v3 (Colab T4)

Trains **velocity Transformer multi-seed + Pose-TGCN** on richer landmarks (`wlasl_landmarks_v4`, **8 train views**; falls back to v3 if v4 missing).

**Recipe:** proven Transformer size from chase v2 (`d_model=128`, 3 layers) + strong TGCN (3 seeds, hidden 128). Beat target: live **64.96%**.

**Runtime → Change runtime type → T4 GPU** before running.

Download `outputs/chase_colab_v3_results.zip` when finished.


In [ ]:
# @title 1) Setup paths
from pathlib import Path

ZIP_PATH = Path('/content/wlasl_colab_chase.zip')
# ZIP_PATH = Path('/content/drive/MyDrive/wlasl_colab_chase.zip')

USE_DRIVE = False
DRIVE_OUT = Path('/content/drive/MyDrive/wlasl_chase_outputs')

WORK = Path('/content/wlasl_colab_chase')
OUT = WORK / 'outputs' / 'chase_colab_v3'
print('ZIP exists:', ZIP_PATH.exists(), ZIP_PATH)
print('OUT:', OUT)


In [ ]:
# @title 2) Optional: mount Drive
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_OUT.mkdir(parents=True, exist_ok=True)
    print("Drive ready:", DRIVE_OUT)

In [ ]:
# @title 3) Unzip package
import zipfile
from pathlib import Path

CONTENT = Path('/content')
WORK = CONTENT / 'wlasl_colab_chase'

meta_v4 = WORK / 'data' / 'processed' / 'wlasl_landmarks_v4' / 'metadata.json'
meta_v3 = WORK / 'data' / 'processed' / 'wlasl_landmarks_v3' / 'metadata.json'

if meta_v4.exists() or meta_v3.exists():
    print('Already unpacked:', WORK)
else:
    assert ZIP_PATH.exists(), f'Upload zip first: {ZIP_PATH}'
    with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
        zf.extractall(CONTENT)
    print('Unpacked to', WORK)

OUT = WORK / 'outputs' / 'chase_colab_v3'
OUT.mkdir(parents=True, exist_ok=True)
LANDMARK_DIR = meta_v4.parent if meta_v4.exists() else meta_v3.parent
print('Using landmarks:', LANDMARK_DIR)
!cat {LANDMARK_DIR}/metadata.json | head -40


In [ ]:
# @title 4) Install deps + verify GPU
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "joblib", "scikit-learn", "numpy"])

import torch
print("torch", torch.__version__)
print("cuda_available", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU — Runtime → Change runtime type → T4 GPU")

In [ ]:
# @title 5) Train chase ensemble v3 (v4 data + proven TF size)
import os, sys, json, time
from pathlib import Path
import torch

os.chdir(WORK)
sys.path.insert(0, str(WORK))

from modules.recognition.wlasl_boost_trainer import train_boosted_landmark_models

meta_v4 = WORK / 'data' / 'processed' / 'wlasl_landmarks_v4' / 'metadata.json'
meta_v3 = WORK / 'data' / 'processed' / 'wlasl_landmarks_v3' / 'metadata.json'
LANDMARK_DIR = meta_v4.parent if meta_v4.exists() else meta_v3.parent
print('LANDMARK_DIR=', LANDMARK_DIR)
print('cuda?', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)

OUT.mkdir(parents=True, exist_ok=True)
t0 = time.time()
report = train_boosted_landmark_models(
    processed_dir=str(LANDMARK_DIR),
    output_dir=str(OUT),
    epochs=140,
    aug_copies=8,
    n_seeds=5,
    train_tgcn_flag=True,
    d_model=128,
    nhead=4,
    layers=3,
    n_tgcn_seeds=3,
    n_tta=9,
    transformer_lr=8e-4,
    tgcn_hidden=128,
    tgcn_stages=14,
)
elapsed = time.time() - t0
print(f'\nDone in {elapsed/60:.1f} min')
print(json.dumps({
    'val_accuracy': report['val_accuracy'],
    'live_baseline_to_beat': 0.6496,
    'landmark_dir': str(LANDMARK_DIR.name),
    'transformer_ensemble': report['transformer_ensemble_val_accuracy'],
    'tgcn': report['tgcn_val_accuracy'],
    'hgb': report['hgb_val_accuracy'],
    'blend_weights': report['blend_weights'],
    'seed_val_accuracies': report['seed_val_accuracies'],
    'n_train': report['n_train'],
    'n_val': report['n_val'],
}, indent=2))


In [ ]:
# @title 6) Package results for download
import json, shutil, zipfile
from pathlib import Path

summary = {
    'run': 'chase_colab_v3',
    'd_model': 128,
    'layers': 3,
    'landmark_dir': str(LANDMARK_DIR),
    'val_accuracy': report['val_accuracy'],
    'num_classes': report['num_classes'],
    'n_train': report['n_train'],
    'n_val': report['n_val'],
    'blend_weights': report['blend_weights'],
    'transformer_ensemble_val_accuracy': report['transformer_ensemble_val_accuracy'],
    'tgcn_val_accuracy': report['tgcn_val_accuracy'],
    'hgb_val_accuracy': report['hgb_val_accuracy'],
    'seed_val_accuracies': report['seed_val_accuracies'],
    'live_baseline_to_beat': 0.6495726495726496,
    'source': 'colab_t4_chase_v3',
}
(OUT / 'colab_summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')

result_zip = WORK / 'outputs' / 'chase_colab_v3_results.zip'
with zipfile.ZipFile(result_zip, 'w', zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(OUT.rglob('*')):
        if p.is_file():
            zf.write(p, arcname=str(Path('chase_colab_v3') / p.relative_to(OUT)))

print('Wrote', result_zip, 'size_mb', round(result_zip.stat().st_size / 1e6, 2))
!ls -lh {OUT}

if USE_DRIVE:
    DRIVE_OUT.mkdir(parents=True, exist_ok=True)
    dest = DRIVE_OUT / 'chase_colab_v3_results.zip'
    shutil.copy2(result_zip, dest)
    print('Copied to Drive:', dest)

try:
    from google.colab import files
    files.download(str(result_zip))
except Exception as e:
    print('Auto-download skipped:', e)
    print('Download manually from', result_zip)
